In [ ]:
import json, os, shutil
from pathlib import Path
import numpy as np, matplotlib as mpl, matplotlib.pyplot as plt
from mtsspi.figures import SANS, SANS_PREAMBLE
mpl.rcParams.update({"text.usetex": False, **SANS, "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 10,
    "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 1.7, "lines.markersize": 2.7, "figure.constrained_layout.use": True,
    "figure.dpi": 180, "savefig.dpi": 600, "savefig.bbox": "tight"})
R = Path(os.environ.get("AXES_RESULTS", "/g/data/ql44/we2614/mts-spi-study/representation/empirical/character-axes-261102"))
OUT = Path(os.environ.get("AXES_FIGURES", R / "figures")); OUT.mkdir(parents=True, exist_ok=True)
C = dict(neg="#7f7f7f", pos="#c0392b", z="#1f77b4", core="#2ca02c", base="#000000")
def save(fig, name):
    for ext in ("svg", "png"): fig.savefig(OUT / f"{name}.{ext}", dpi=180 if ext == "png" else None)
def short(n):
    for a, b in (("_multitaper_mean_fs-1", ""), ("_EmpiricalCovariance", ""), ("_fmin-0-25_fmax-0-5", " H"), ("_fmin-0_fmax-0-25", " L"),
                 ("_fmin-0_fmax-0-5", ""), ("_fmin-1e-05", ""), ("_k-1_kt-1_l-1_lt-1", ""), ("_NN-4", ""), ("_DCE-AUTO", "·DCE")):
        n = n.replace(a, b)
    return n
load = lambda p: json.load(open(p)) if Path(p).exists() else None

In [ ]:
# Overview: what each feature set achieves (AUROC); P = Pearson + strength, S = P + channel spectra, Q = P + channel value shape
contrast = dict(neurotycho="anaesthesia_vs_awake", propofol="moderate_vs_baseline", lemon="old_vs_young", ad_ftd="AD_vs_control")
sets = [("P", "Pearson+strength"), ("P_core", "+ core"), ("P_z", "+ z"), ("S", "+ channel spectra"), ("S_core", "+ spectra + core"),
        ("S_z", "+ spectra + z"), ("Q", "+ channel value shape"), ("Q_z", "+ value shape + z")]
fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.6), sharey=True)
for ax, (cohort, label) in zip(axes, contrast.items()):
    s = load(R / f"sufficiency/core-v3b/{cohort}.json"); b = load(R / f"beyond-channels/{cohort}.json")
    sc = {**(s["contrasts"][label]["scores"] if s else {}), **(b["contrasts"][label]["scores"] if b else {})}
    for i, (k, name) in enumerate(sets):
        if k in sc:
            col = C["z"] if k.endswith("_z") else C["core"] if "core" in k else C["base"]
            ax.plot(sc[k]["auroc"], i, "o", color=col, ms=4)
    ax.axvline(.5, color="0.8", lw=.8); ax.set_xlim(.4, 1.02); ax.set_title(cohort.replace("_", "/"), fontsize=9); ax.set_xlabel("AUROC")
axes[0].set_yticks(range(len(sets)), [n for _, n in sets]); axes[0].invert_yaxis()
save(fig, "overview_auroc")

In [ ]:
# Core readouts per unit (paired lines): what changes, by named character readout
fig, rows = plt.subplots(2, 9, figsize=(7.2, 2.9))
for row, cohort in zip(rows, ("neurotycho", "propofol")):
    D = np.load(R / f"figure-data/{cohort}.npz")
    for ax, (k, name) in zip(row, enumerate(D["core_names"])):
        v = D["core_units"][:, :, k]
        ax.plot([0, 1], v, color="0.6", lw=.8); ax.plot([0, 1], np.nanmean(v, 1), color=C["pos"], lw=2)
        ax.set_xticks([0, 1], D["classes"], rotation=45, fontsize=6); ax.tick_params(labelsize=6)
        ax.set_title(f"{name}\n({D['core_roles'][k]})", fontsize=6)
    for ax in row[len(D["core_names"]):]: ax.axis("off")
    row[0].set_ylabel(cohort, fontsize=8)
save(fig, "core_readouts_units")

In [ ]:
# z landscape: paired effect of every SPI-SPI coordinate, and the strongest unanimous pairs
fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.2), gridspec_kw=dict(width_ratios=[1, 1.6]))
for (h, bar), cohort in zip(axes, ("neurotycho", "propofol")):
    D = np.load(R / f"figure-data/{cohort}.npz"); d = D["d"]
    h.hist(d[np.isfinite(d)], bins=120, color="0.5"); h.set_xlabel("paired d (all 39,903 coordinates)"); h.set_title(cohort, fontsize=9)
    top = D["top"]; bar.barh(range(len(top)), d[top], color=np.where(d[top] > 0, C["pos"], C["z"]))
    bar.set_yticks(range(len(top)), [short(D["names_z"][k]) for k in top], fontsize=6); bar.invert_yaxis()
    bar.axvline(0, color="0.6", lw=.8); bar.set_xlabel(f"paired d ({D['classes'][1]} − {D['classes'][0]}), all units agree")
save(fig, "z_landscape")

In [ ]:
# Deep dive: the top pairs traced to channel pairs (unit-mean maps per class) and their edge agreement (z)
for cohort, npairs in (("neurotycho", 2), ("propofol", 1)):
    D = np.load(R / f"figure-data/{cohort}.npz")
    fig, axes = plt.subplots(npairs, 5, figsize=(7.2, 1.7 * npairs), squeeze=False)
    for row, (t, k) in zip(axes, enumerate(D["top"][:npairs])):
        a, b = D["names_z"][k].split("|"); M = int(D["M"]); off = ~np.eye(M, dtype=bool)
        for j, (s, ci) in enumerate(((0, 0), (0, 1), (1, 0), (1, 1))):
            m = D["maps"][t, s, ci]; lim = np.nanpercentile(np.abs(D["maps"][t, s]), 98)
            row[j].imshow(m, cmap="viridis", vmin=np.nanmin(D["maps"][t, s]), vmax=np.nanmax(D["maps"][t, s]))
            row[j].set_title(f"{short([a, b][s])}\n{D['classes'][ci]}", fontsize=6); row[j].set_xticks([]); row[j].set_yticks([])
        for ci, col in ((0, C["neg"]), (1, C["pos"])):
            x, y = D["maps"][t, 0, ci][off], D["maps"][t, 1, ci][off]
            row[4].plot(x, y, "o", color=col, ms=1.5, alpha=.6, label=f"{D['classes'][ci]}: z̄ {D['z_class'][ci, k]:.2f}")
        row[4].set_xlabel(short(a), fontsize=6); row[4].set_ylabel(short(b), fontsize=6); row[4].legend(fontsize=6)
    save(fig, f"deep_dive_{cohort}")